# 02 · Signal: people, code, tools, and judges

Every verdict is an MLflow **assessment** on a trace: feedback (good/bad, with a rationale) or an expectation
(the right answer). There are four kinds of signal, from most trusted to most scalable:

| Source | Example | Cost | Trust |
|---|---|---|---|
| people | reviewers' `quality` labels + `expected_response` | expensive | ground truth, but reviewers disagree |
| code | `no_stale_policy`, `no_disclaimer_opener` | free | exact, narrow |
| tool-level | `read_right_policy`, `focused_reading` (from TOOL spans) | free | judges *how* the answer was found |
| LLM judges | `Guidelines`, `make_judge("quality")` | model calls | flexible; must be checked against people |

In [ ]:
from sia import notebook as nb

nb.setup(backend="offline")

In [ ]:
# Prerequisite (idempotent): v1 in production and a "week" of labeled production traffic.
from sia import tracking, traffic

nb.ensure_production()
if len(tracking.agent_traces()) < 12:
    ids = traffic.send(("traffic",), n=16, seed=1)        # simulated customers -> production traces
    traffic.label(ids, rate=1.0, seed=1)                 # simulated reviewers label every one
print(len(tracking.agent_traces()), "production traces")

## People: three reviewers, three rubrics

The simulated reviewers share an answer key but grade differently:
* **maria** checks facts only;
* **dev** also fails disclaimers;
* **sam** also fails long answers.

Real reviewer pools behave like this, which is why a test generated from one person's label can encode their
personal rubric.

In [ ]:
from sia.reviewers import review
from sia.scenario import load_cases

case = next(c for c in load_cases() if c.id == "refund_window")
answer = "Policies can vary, so double-check the Terms of Service. Refunds are available for 14 days."
for r in ("maria", "dev", "sam"):
    v = review(r, case, answer)
    print(f"{r:<6} {'PASS' if v.passed else 'FAIL'}  {v.rationale[:100]}")

What a labeled production trace carries. These assessments were written by `log_feedback` and
`log_expectation` in `sia.reviewers.log_review`.

In [ ]:
import mlflow

t = tracking.agent_traces()[0]
print(tracking.question_of(t), "->", tracking.response_of(t)[:120], "\n")
for a in mlflow.get_trace(t.info.trace_id).info.assessments:
    kind = "feedback" if a.feedback is not None else "expectation"
    print(f"{kind:<11} {a.name:<18} {str(a.value)[:60]:<60} by {a.source.source_id}")

## Code and tool-level checks, run *on the existing traces*

`mlflow.genai.evaluate()` with traces as data and no `predict_fn` doesn't re-run the agent. It attaches each
scorer's verdict to the trace, next to the human labels.

In [ ]:
import pandas as pd
from sia import signals

traces = tracking.agent_traces()
result = mlflow.genai.evaluate(data=traces, scorers=signals.code_scorers())
cols = [c for c in result.result_df.columns if c.endswith("/value") and not c.startswith(("expected", "key_facts", "quality"))]
summary = result.result_df[cols].apply(lambda s: s.map(lambda v: str(v).lower() in ("true", "yes")).mean())
summary.rename(lambda c: c.removesuffix("/value")).to_frame("pass rate").style.format("{:.0%}")

v1 never opens the knowledge files, so `read_right_policy` fails everywhere. Here's how that check reads the
trace:

In [ ]:
print(nb.source(signals, "read_right_policy"))

## LLM judges

Two kinds:
* `Guidelines` scorers are pass/fail requirements written in plain English.
* `quality` is a `make_judge` judge named like the human feedback, so Level 4 can **align** it to the reviewers.

These need a model, so this cell runs only when one is configured (by default Claude Opus 5.5, with
`ANTHROPIC_API_KEY` set).

In [ ]:
from sia import config

print(signals.leads_with_answer.guidelines, "\n")
print(signals.QUALITY_INSTRUCTIONS)
if config.HAS_LLM:
    t = traces[0]
    fb = signals.quality_judge()(trace=t)
    print("quality judge says:", fb.value, "|", (fb.rationale or "")[:200])
else:
    print("(no LLM configured in this notebook: judge cell skipped)")

## Can we trust the verifiers?

Before any automated signal drives an update, check it. Reviewer strictness is one view; judge-vs-human agreement
(Level 1, `03_signal_quality.py`) is the other.

In [ ]:
rows = []
for t in traces:
    fb = tracking.human_feedback(t)
    if fb:
        rows.append({"reviewer": fb.source.source_id, "passed": bool(fb.value)})
pd.DataFrame(rows).groupby("reviewer")["passed"].agg(labeled="count", pass_rate="mean").style.format({"pass_rate": "{:.0%}"})

## Signal is queryable

Boolean feedback is stored as text, so the filter is `= 'false'`, not `= False`.

In [ ]:
failing = tracking.agent_traces("feedback.quality = 'false'")
by_topic = pd.Series([t.info.tags.get("topic") for t in failing]).value_counts()
print(f"{len(failing)} human-failed traces"); by_topic

**Next:** 03 · Store, update, gate. Turning this signal into a shipped change.